# BioMQM — Baseline Pipeline

Full end-to-end pipeline for the BioMQM baseline:
1. **QG** — Question Generation (calls `QG/code/qwen-3b.py`)
2. **QA** — Question Answering on source + BT (calls `QA/code/qwen-3b.py`)
3. **Mapping** — Map QG ↔ QA source ↔ QA BT
4. **String Comparison** — F1, EM, chrF, BLEU
5. **SBERT** — Sentence-BERT cosine similarity
6. **Desiderata** — Quality analysis

| Parameter | Values |
|-----------|--------|
| Model | Qwen/Qwen2.5-3B-Instruct (swap script in `QG/code/` and `QA/code/` for a different model) |
| Languages | de, es, fr, ru, zh-CN |

## 0. Environment Setup

In [ ]:
import os, sys, subprocess, json

IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = os.path.exists('/kaggle')
print(f"Environment: {'Kaggle' if IN_KAGGLE else 'Colab' if IN_COLAB else 'Local'}")

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_CACHE_DIR = '/content/drive/MyDrive/AskQE_Models_Cache'
    os.makedirs(DRIVE_CACHE_DIR, exist_ok=True)
    os.environ['HF_HOME'] = DRIVE_CACHE_DIR
    os.environ['TRANSFORMERS_CACHE'] = os.path.join(DRIVE_CACHE_DIR, 'transformers')
    os.environ['SENTENCE_TRANSFORMERS_HOME'] = os.path.join(DRIVE_CACHE_DIR, 'sentence_transformers')

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers', 'torch', 'accelerate', 'nltk',
                'sentence-transformers', 'sacrebleu', 'textstat'], check=True)
print('Dependencies installed!')

In [ ]:
if IN_KAGGLE:
    PROJECT_ROOT = '/kaggle/working/askqe'
elif IN_COLAB:
    PROJECT_ROOT = '/content/askqe'
else:
    PROJECT_ROOT = os.getcwd()

if not os.path.exists(PROJECT_ROOT) and (IN_KAGGLE or IN_COLAB):
    subprocess.run(['git', 'clone',
                    'https://github.com/Simone280802/AskQE_DNLP_2025-2026.git',
                    PROJECT_ROOT], check=True)

print(f'Project root: {PROJECT_ROOT}')

## 1. Configuration

To use a different model, change `QG_SCRIPT` and `QA_SCRIPT` to point to the model-specific scripts in `QG/code/` and `QA/code/`.

In [ ]:
# ═══════════════════════════════════════════════════
# CHANGE THESE TO USE A DIFFERENT MODEL
# Available: qwen-3b, gemma-9b, gemma-27b, llama-8b, llama-70b, yi-9b
# ═══════════════════════════════════════════════════
MODEL_SHORT = 'qwen-3b'
QG_SCRIPT   = f'{PROJECT_ROOT}/QG/code/{MODEL_SHORT}.py'
QA_SCRIPT   = f'{PROJECT_ROOT}/QA/code/{MODEL_SHORT}.py'

BASELINE_DIR = f'{PROJECT_ROOT}/Qwen2.5-3B-Instruct/biomqm/baseline'
EVAL_DIR     = f'{BASELINE_DIR}/evaluation'
DATA_DIR     = f'{PROJECT_ROOT}/biomqm'
LANGUAGES    = ['de', 'es', 'fr', 'ru', 'zh-CN']

# Create output directories
for d in ['QG', 'QA/source', 'QA/bt', 'mapping',
          'evaluation/sbert', 'evaluation/string comparison',
          'evaluation/desiderata']:
    os.makedirs(f'{BASELINE_DIR}/{d}', exist_ok=True)

print(f'Model: {MODEL_SHORT}')
print(f'QG script: {QG_SCRIPT}')
print(f'QA script: {QA_SCRIPT}')

## 2. Pre-download Models

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
import torch

MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'
print(f'[1/2] Caching {MODEL_ID}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, device_map='auto')
del model, tokenizer
torch.cuda.empty_cache() if torch.cuda.is_available() else None
print('      ✓ LLM cached')

print('[2/2] Caching SBERT...')
_ = SentenceTransformer('all-MiniLM-L6-v2')
del _
print('      ✓ SBERT cached')

## 3. QG — Question Generation

Calls `QG/code/{model}.py` which uses prompts from `QG/code/prompt.py`.

In [ ]:
qg_output = f'{BASELINE_DIR}/QG/{MODEL_SHORT}.jsonl'

cmd = [
    sys.executable, '-u', QG_SCRIPT,
    '--output_path', qg_output,
    '--prompt', 'vanilla'
]

print(f'Running QG (vanilla prompt) → {qg_output}')
subprocess.run(cmd, check=True)
print('✓ QG complete!')

## 4. QA — Question Answering

Calls `QA/code/{model}.py` which uses the QA prompt from `QA/code/prompt.py`.

### 4a. QA Source

In [ ]:
qa_src_output = f'{BASELINE_DIR}/QA/source-vanilla.jsonl'

cmd = [
    sys.executable, '-u', QA_SCRIPT,
    '--qg_input_path', qg_output,
    '--output_path', qa_src_output,
    '--sentence_type', 'en'
]

print(f'Running QA Source → {qa_src_output}')
subprocess.run(cmd, check=True)
print('✓ QA Source complete!')

### 4b. QA BT (all languages)

In [ ]:
for lang in LANGUAGES:
    qa_bt_output = f'{BASELINE_DIR}/QA/bt-{lang}-vanilla.jsonl'

    cmd = [
        sys.executable, '-u', QA_SCRIPT,
        '--qg_input_path', qg_output,
        '--output_path', qa_bt_output,
        '--sentence_type', 'bt',
        '--lang', lang
    ]

    print(f'Running QA BT [{lang}]...')
    subprocess.run(cmd, check=True)
    print(f'✓ QA BT [{lang}] complete!')

print('\n✓ All QA complete!')

## 5. Mapping

In [ ]:
mapping_script = f'{BASELINE_DIR}/mapping/mapping.py'
cmd = [sys.executable, '-u', mapping_script,
       '--base_dir', BASELINE_DIR]
print('Running mapping...')
subprocess.run(cmd, check=True)
print('✓ Mapping complete!')

## 6. String Comparison

In [ ]:
str_script = f'{EVAL_DIR}/string comparison/string_comparison.py'
cmd = [sys.executable, '-u', str_script]
print('Running string comparison...')
subprocess.run(cmd, check=True)
print('✓ String comparison complete!')

## 7. SBERT Evaluation

In [ ]:
sbert_script = f'{EVAL_DIR}/sbert/sbert.py'
cmd = [sys.executable, '-u', sbert_script]
print('Running SBERT evaluation...')
subprocess.run(cmd, check=True)
print('✓ SBERT evaluation complete!')

## 8. Desiderata Analysis

In [ ]:
desiderata_dir = f'{EVAL_DIR}/desiderata'
for script in ['i_avg_questions.py', 'i_diversity.py', 'i_duplicate.py',
               'q_answerability.py', 'q_readability.py']:
    path = os.path.join(desiderata_dir, script)
    if os.path.exists(path):
        print(f'Running {script}...')
        subprocess.run([sys.executable, '-u', path], check=True)
        print(f'  ✓ {script} done')
    else:
        print(f'  ⚠ {script} not found, skipping')

print('\n✓ Desiderata analysis complete!')

## Summary

Full baseline pipeline complete! Output structure:
```
baseline/
├── QG/{model}.jsonl
├── QA/source-vanilla.jsonl
├── QA/bt-{lang}-vanilla.jsonl
├── mapping/mapping.jsonl
└── evaluation/
    ├── sbert/
    ├── string comparison/
    └── desiderata/
```